In [1]:
# Cell 1: Path Fix + Imports
import sys
import os
sys.path.append(os.path.abspath('.'))

import json
from src.retrieval import retriever
from src.evaluation import load_evaluation_set

# For Flan-T5
from transformers import T5ForConditionalGeneration, T5Tokenizer

# For RoBERTa Extractive
from transformers import pipeline

print("All imports successful!")

ModuleNotFoundError: No module named 'src'

In [ ]:
# Cell 2: Load Models

# 1. Grok (already in generation.py)
from src.generation import generate_answer as grok_answer

# 2. Flan-T5 (Encoder-Decoder)
t5_tokenizer = T5Tokenizer.from_pretrained("google/flan-t5-base")
t5_model = T5ForConditionalGeneration.from_pretrained("google/flan-t5-base")

# 3. RoBERTa Extractive QA
qa_pipeline = pipeline("question-answering", model="deepset/roberta-base-squad2")

In [ ]:
# Cell 3: Run Comparison on Sample Questions
eval_set = load_evaluation_set()[:8]  # First 8 questions for speed

results = []

for q in eval_set:
    question = q["question"]
    
    # Grok Answer
    grok_res = grok_answer(question, retriever)
    grok_ans = grok_res["answer"]
    
    # Flan-T5 Answer
    input_text = f"question: {question} context: {' '.join([c['content'] for c in grok_res['retrieved_chunks']])}"
    inputs = t5_tokenizer(input_text, return_tensors="pt", max_length=512, truncation=True)
    outputs = t5_model.generate(**inputs, max_new_tokens=150)
    t5_ans = t5_tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # RoBERTa Extractive
    context = " ".join([c['content'] for c in grok_res['retrieved_chunks']])
    try:
        roberta_res = qa_pipeline(question=question, context=context)
        roberta_ans = roberta_res['answer']
    except:
        roberta_ans = "No answer"
    
    results.append({
        "question": question,
        "grok": grok_ans[:150],
        "flan_t5": t5_ans[:150],
        "roberta": roberta_ans[:150]
    })

# Save results
import pandas as pd
df = pd.DataFrame(results)
df.to_csv("outputs/model_comparison.csv", index=False)
df